In [2]:

import pandas as pd
from pathlib import Path

# Location of the original datasets
data_path = Path(
    "Topic3_Climate_Energy_The_Green_Tech_Carbon_Spike/data"
)

# Load the datasets
grid = pd.read_csv(data_path / "grid_hourly.csv")
weather = pd.read_csv(data_path / "weather_hourly.csv")

# Record original row counts
print("Original grid rows:", len(grid))
print("Original weather rows:", len(weather))

# Preview both datasets
display(grid.head())
display(weather.head())


Original grid rows: 17520
Original weather rows: 17520


,utc_time_at_end_of_hour,balancing_authority,demand_forecast_mwh,demand_mwh,net_generation,total_interchange,coal,natural_gas,nuclear,wind,solar,hydro,other,unit
0,2022-01-01 07:00,MCPA,634,643,643.0,0.0,284.0,0.0,211.0,106.0,0.0,32.0,10.0,NaN
1,2022-01-01 08:00,MCPA,633,642,642.0,0.0,282.0,0.0,210.0,106.0,0.0,34.0,10.0,NaN
2,2022-01-01 09:00,MCPA,639,641,640.0,0.0,286.0,0.0,209.0,106.0,0.0,30.0,9.0,NaN
3,2022-01-01 10:00,MCPA,638,646,646.0,0.0,302.0,0.0,210.0,94.0,0.0,30.0,10.0,NaN
4,2022-01-01 11:00,MCPA,645,654,654.0,0.0,332.0,0.0,211.0,73.0,0.0,28.0,10.0,NaN


,timestamp,station_id,temp_f,heating_degree_hours,wind_speed_mph,relative_humidity
0,2022-01-01 00:00,WX-0417,40.1,24.9,5,72
1,2022-01-01 01:00,WX-0417,38.1,26.9,4,83
2,2022-01-01 02:00,WX-0417,36.9,28.1,4,88
3,2022-01-01 03:00,WX-0417,35.4,29.6,4,93
4,2022-01-01 04:00,WX-0417,34.8,30.2,4,86


In [ ]:
import pandas as pd
from pathlib import Path

data_path = Path("Topic3_Climate_Energy_The_Green_Tech_Carbon_Spike/data")

grid = pd.read_csv(data_path / "grid_hourly.csv")

print("Original grid rows:", len(grid))
print("\nColumn names:")
print(grid.columns.tolist())

display(grid.head())

In [ ]:

# Check for missing values
print("MISSING VALUES:")
print(grid.isnull().sum())

# Check for duplicate rows
print("\nDUPLICATE ROWS:")
print(grid.duplicated().sum())

# Check for negative electricity demand
print("\nNEGATIVE DEMAND VALUES:")
print((grid["demand_mwh"] < 0).sum())

# Check the different units
print("\nUNIT LABELS:")
print(grid["unit"].value_counts(dropna=False))

# Total rows before cleaning
print("\nTOTAL ROWS:")
print(len(grid))


In [ ]:

# Find all rows with negative electricity demand
negative_demand = grid[grid["demand_mwh"] < 0]

print("Number of negative demand rows:", len(negative_demand))

# Display the affected records
display(negative_demand[
    ["utc_time_at_end_of_hour",
     "demand_mwh",
     "net_generation",
     "unit"]
])


In [ ]:

# Compare negative demand with forecasts
negative_demand = grid[grid["demand_mwh"] < 0].copy()

negative_demand["absolute_demand"] = (
    negative_demand["demand_mwh"].abs()
)

display(negative_demand[
    ["utc_time_at_end_of_hour",
     "demand_forecast_mwh",
     "demand_mwh",
     "absolute_demand",
     "net_generation"]
])


In [ ]:

# Create a cleaned copy of the grid data
grid_clean = grid.copy()

# Count negative values before correction
negative_count = (grid_clean["demand_mwh"] < 0).sum()

# Correct negative demand values
grid_clean["demand_mwh"] = grid_clean["demand_mwh"].abs()

# Verify the correction
print("Original rows:", len(grid))
print("Negative values corrected:", negative_count)
print("Rows removed:", len(grid) - len(grid_clean))
print("Remaining rows:", len(grid_clean))
print("Negative values remaining:",
      (grid_clean["demand_mwh"] < 0).sum())


In [ ]:

# Find rows with missing wind generation
missing_wind = grid_clean[grid_clean["wind"].isna()].copy()

print("Missing wind values:", len(missing_wind))

# Display the first 10 affected records
display(missing_wind[
    ["utc_time_at_end_of_hour",
     "net_generation",
     "coal",
     "natural_gas",
     "nuclear",
     "wind",
     "solar",
     "hydro",
     "other"]
].head(10))


In [ ]:

# Generation sources
sources = [
    "coal", "natural_gas", "nuclear",
    "wind", "solar", "hydro", "other"
]

# Check rows where wind is already known
known_wind = grid_clean[grid_clean["wind"].notna()].copy()

# Calculate the sum of generation sources
known_wind["source_total"] = known_wind[sources].sum(axis=1)

# Compare against reported net generation
known_wind["difference"] = (
    known_wind["net_generation"] - known_wind["source_total"]
)

print("Rows checked:", len(known_wind))
print("Exact matches:", (known_wind["difference"] == 0).sum())
print("Within 2 MWh:", (known_wind["difference"].abs() <= 2).sum())
print("Largest difference:", known_wind["difference"].abs().max())

display(known_wind[
    ["net_generation", "source_total", "difference"]
].head(10))


In [ ]:

# List generation sources other than wind
other_sources = [
    "coal", "natural_gas", "nuclear",
    "solar", "hydro", "other"
]

# Find rows with missing wind values
missing_wind = grid_clean[grid_clean["wind"].isna()].copy()

# Calculate missing wind generation
missing_wind["calculated_wind"] = (
    missing_wind["net_generation"]
    - missing_wind[other_sources].sum(axis=1)
)

# Check calculated values
print("Missing wind values:", len(missing_wind))
print("Negative calculated values:",
      (missing_wind["calculated_wind"] < 0).sum())
print("Minimum calculated wind:",
      missing_wind["calculated_wind"].min())
print("Maximum calculated wind:",
      missing_wind["calculated_wind"].max())

display(missing_wind[
    ["utc_time_at_end_of_hour", "calculated_wind"]
].head(10))


In [ ]:

# Count missing values before filling
missing_before = grid_clean["wind"].isna().sum()

# Fill missing wind values using calculated values
grid_clean.loc[missing_wind.index, "wind"] = (
    missing_wind["calculated_wind"]
)

# Verify the results
print("Wind values filled:", missing_before)
print("Missing wind values remaining:",
      grid_clean["wind"].isna().sum())
print("Rows remaining:", len(grid_clean))


In [ ]:
# Inspect original generation values by recorded unit (before conversion)
print("ORIGINAL UNIT LABEL COUNTS:")
print(grid["unit"].value_counts(dropna=False).to_string())
print("\nNET GENERATION BY ORIGINAL UNIT:")
display(grid.groupby("unit", dropna=False)["net_generation"].agg(["count", "min", "median", "max"]))


In [ ]:
# Convert generation and interchange quantities into MWh.
# Data dictionary: unit applies to net_generation, total_interchange, and fuel columns;
# demand_mwh and demand_forecast_mwh are already in MWh.
generation_columns = [
    "net_generation", "total_interchange", "coal", "natural_gas",
    "nuclear", "wind", "solar", "hydro", "other"
]
gwh_mask = grid_clean["unit"].eq("GWh")
missing_units = grid_clean["unit"].isna().sum()
gwh_rows = gwh_mask.sum()

# GWh values are 1/1000 of comparable MWh values in the source data.
grid_clean.loc[gwh_mask, generation_columns] *= 1000
# The blank-unit rows are on the MWh scale; standardize their metadata too.
grid_clean["unit"] = "MWh"

print("GWh rows converted to MWh:", gwh_rows)
print("Blank unit labels standardized to MWh:", missing_units)
print("Remaining missing unit labels:", grid_clean["unit"].isna().sum())
print("Units after conversion:")
print(grid_clean["unit"].value_counts().to_string())
print("\nNet generation after conversion:")
print(grid_clean["net_generation"].agg(["min", "median", "max"]))


In [ ]:

# Final grid validation
print("Total rows:", len(grid_clean))

print("\nMissing values:")
print(grid_clean.isna().sum().to_string())

print("\nNegative demand values:",
      (grid_clean["demand_mwh"] < 0).sum())

print("Duplicate timestamps:",
      grid_clean["utc_time_at_end_of_hour"].duplicated().sum())

print("\nDate range:")
print("First:", grid_clean["utc_time_at_end_of_hour"].min())
print("Last:", grid_clean["utc_time_at_end_of_hour"].max())


In [ ]:

# Check weather dataset for problems
print("Original weather rows:", len(weather))

print("\nMissing values:")
print(weather.isna().sum().to_string())

print("\nDuplicate rows:", weather.duplicated().sum())

print("Duplicate timestamps:",
      weather["timestamp"].duplicated().sum())

print("\nDate range:")
print("First:", weather["timestamp"].min())
print("Last:", weather["timestamp"].max())


In [ ]:

# Find weather records with repeated timestamps
duplicate_times = weather[
    weather["timestamp"].duplicated(keep=False)
].copy()

print("Records with repeated timestamps:", len(duplicate_times))

# Display the affected records
display(duplicate_times.sort_values("timestamp"))


In [ ]:

# Create a cleaned copy of the weather dataset
weather_clean = weather.copy()

# Check missing temperature and heating degree hours
missing_temp = weather_clean["temp_f"].isna()
missing_heating = weather_clean["heating_degree_hours"].isna()

print("Missing temperatures:", missing_temp.sum())
print("Missing heating degree hours:", missing_heating.sum())

print("Rows missing both:",
      (missing_temp & missing_heating).sum())

print("Rows missing only temperature:",
      (missing_temp & ~missing_heating).sum())

print("Rows missing only heating degree hours:",
      (~missing_temp & missing_heating).sum())

# Display first 10 affected records
display(weather_clean[
    missing_temp | missing_heating
].head(10))


In [ ]:

# Identify consecutive groups of missing temperatures
missing = weather_clean["temp_f"].isna()

group_ids = (missing != missing.shift()).cumsum()

missing_groups = (
    weather_clean[missing]
    .groupby(group_ids[missing])
    .size()
)

print("Total missing temperatures:", missing.sum())
print("Number of missing groups:", len(missing_groups))
print("Longest consecutive missing group:", missing_groups.max())

print("\nGroup sizes:")
print(missing_groups.value_counts().sort_index())


In [ ]:

# Count missing temperatures before cleaning
missing_before = weather_clean["temp_f"].isna().sum()

# Fill missing temperatures using linear interpolation
weather_clean["temp_f"] = (
    weather_clean["temp_f"].interpolate(
        method="linear",
        limit=1,
        limit_area="inside"
    )
)

# Verify the results
print("Temperatures filled:", missing_before)
print("Missing temperatures remaining:",
      weather_clean["temp_f"].isna().sum())
print("Rows remaining:", len(weather_clean))


In [ ]:

# Check the heating degree hour formula
known = weather_clean["heating_degree_hours"].notna()

calculated = (
    65 - weather_clean.loc[known, "temp_f"]
).clip(lower=0)

difference = (
    weather_clean.loc[known, "heating_degree_hours"]
    - calculated
).abs()

print("Existing records checked:", known.sum())
print("Records matching formula:", (difference < 0.01).sum())
print("Largest difference:", difference.max())
print("Missing heating degree hours:",
      weather_clean["heating_degree_hours"].isna().sum())


In [ ]:

# Identify missing heating degree hours
missing_heating = weather_clean["heating_degree_hours"].isna()

# Calculate missing values from temperature
weather_clean.loc[missing_heating, "heating_degree_hours"] = (
    65 - weather_clean.loc[missing_heating, "temp_f"]
).clip(lower=0)

# Verify the results
print("Heating degree hours filled:", missing_heating.sum())
print("Missing values remaining:",
      weather_clean["heating_degree_hours"].isna().sum())
print("Rows remaining:", len(weather_clean))


In [ ]:

# Final weather validation
print("Total rows:", len(weather_clean))

print("\nMissing values:")
print(weather_clean.isna().sum().to_string())

print("\nTemperature range:")
print("Minimum:", weather_clean["temp_f"].min())
print("Maximum:", weather_clean["temp_f"].max())

print("\nHumidity range:")
print("Minimum:", weather_clean["relative_humidity"].min())
print("Maximum:", weather_clean["relative_humidity"].max())

print("\nNegative wind speeds:",
      (weather_clean["wind_speed_mph"] < 0).sum())

print("Duplicate timestamps:",
      weather_clean["timestamp"].duplicated().sum())


In [ ]:

# Display grid timestamps around daylight saving time
grid_times = pd.to_datetime(
    grid_clean["utc_time_at_end_of_hour"]
)

weather_times = pd.to_datetime(
    weather_clean["timestamp"]
)

print("GRID - November 6, 2022:")
display(grid_clean.loc[
    grid_times.between("2022-11-06 05:00", "2022-11-06 10:00"),
    ["utc_time_at_end_of_hour", "demand_mwh"]
])

print("\nWEATHER - November 6, 2022:")
display(weather_clean.loc[
    weather_times.between("2022-11-06 00:00", "2022-11-06 04:00"),
    ["timestamp", "temp_f"]
])


In [ ]:

# Convert weather timestamps to UTC
weather_clean["local_time"] = pd.to_datetime(
    weather_clean["timestamp"]
)

weather_clean["weather_utc"] = (
    weather_clean["local_time"]
    .dt.tz_localize(
        "America/Chicago",
        ambiguous="infer",
        nonexistent="NaT"
    )
    .dt.tz_convert("UTC")
)

# Check the converted timestamps
print("Missing UTC timestamps:",
      weather_clean["weather_utc"].isna().sum())

print("Duplicate UTC timestamps:",
      weather_clean["weather_utc"].duplicated().sum())

print("\nConverted daylight saving timestamps:")
display(weather_clean.loc[
    weather_clean["timestamp"].isin([
        "2022-11-06 01:00",
        "2023-11-05 01:00"
    ]),
    ["timestamp", "weather_utc"]
])


In [ ]:

# Convert grid timestamps to UTC
grid_clean["grid_utc"] = pd.to_datetime(
    grid_clean["utc_time_at_end_of_hour"],
    utc=True
)

# Check alignment with weather timestamps
grid_hours = set(grid_clean["grid_utc"])
weather_hours = set(weather_clean["weather_utc"])

# Option 1: Match timestamps directly
direct_matches = len(grid_hours & weather_hours)

# Option 2: Shift weather timestamps forward one hour
shifted_weather_hours = {
    t + pd.Timedelta(hours=1)
    for t in weather_hours
}

shifted_matches = len(grid_hours & shifted_weather_hours)

print("Direct timestamp matches:", direct_matches)
print("Matches with weather shifted +1 hour:", shifted_matches)
print("Total grid records:", len(grid_clean))
print("Total weather records:", len(weather_clean))


In [ ]:

# Align weather timestamps with grid hour-ending timestamps
weather_clean["matching_utc"] = (
    weather_clean["weather_utc"] + pd.Timedelta(hours=1)
)

# Merge grid and weather datasets
combined_hourly = pd.merge(
    grid_clean,
    weather_clean,
    left_on="grid_utc",
    right_on="matching_utc",
    how="inner",
    validate="one_to_one"
)

# Verify the merge
print("Original grid rows:", len(grid_clean))
print("Original weather rows:", len(weather_clean))
print("Combined hourly rows:", len(combined_hourly))
print("Missing values:", combined_hourly.isna().sum().sum())
print("Duplicate UTC timestamps:",
      combined_hourly["grid_utc"].duplicated().sum())

# Preview combined data
display(combined_hourly[
    ["grid_utc", "demand_mwh", "temp_f",
     "wind", "coal", "natural_gas"]
].head())


In [ ]:

# Use the weather observation's local calendar date
combined_hourly["date"] = (
    pd.to_datetime(combined_hourly["timestamp"]).dt.date
)

# Columns to average for each day
average_columns = [
    "demand_mwh",
    "demand_forecast_mwh",
    "net_generation",
    "coal",
    "natural_gas",
    "nuclear",
    "wind",
    "solar",
    "hydro",
    "other",
    "temp_f",
    "heating_degree_hours",
    "wind_speed_mph",
    "relative_humidity"
]

# Calculate daily averages and count hourly records
daily_averages = (
    combined_hourly.groupby("date")
    .agg(
        **{col: (col, "mean") for col in average_columns},
        hours_recorded=("grid_utc", "count")
    )
    .reset_index()
)

# Check results
print("Original hourly records:", len(combined_hourly))
print("Daily records:", len(daily_averages))

print("\nHours per day:")
print(daily_averages["hours_recorded"].value_counts().sort_index())

display(daily_averages.head())


In [ ]:

# Verify daily aggregation
print("Daily records:", len(daily_averages))

print("Total hours accounted for:",
      daily_averages["hours_recorded"].sum())

print("Original hourly records:",
      len(combined_hourly))

print("Missing daily values:",
      daily_averages.isna().sum().sum())

print("Duplicate dates:",
      daily_averages["date"].duplicated().sum())

print("\nDate range:")
print("First:", daily_averages["date"].min())
print("Last:", daily_averages["date"].max())

# Check that the daily averages preserve the overall demand
weighted_demand = (
    (daily_averages["demand_mwh"] *
     daily_averages["hours_recorded"]).sum()
    / daily_averages["hours_recorded"].sum()
)

print("\nOriginal hourly demand average:",
      combined_hourly["demand_mwh"].mean())

print("Weighted daily demand average:",
      weighted_demand)


In [1]:

# Save cleaned datasets to the project data folder
output_folder = data_path

# Save hourly dataset
combined_hourly.to_csv(
    output_folder / "combined_hourly_clean.csv",
    index=False
)

# Save daily averages
daily_averages.to_csv(
    output_folder / "combined_daily_averages.csv",
    index=False
)

# Confirm saved files
print("Saved combined_hourly_clean.csv")
print("Saved combined_daily_averages.csv")
print("Hourly rows:", len(combined_hourly))
print("Daily rows:", len(daily_averages))


NameError: name 'data_path' is not defined